1.Upload the data from /datasets/travel_insurance_us.csv to the data variable. Print the first ten rows on the screen. Examine the data.

In [10]:
#Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 

# Step 2 Ml processing 
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder # for label encoding


In [11]:
data = pd.read_csv('/home/susan/Feature_Engineering_Sprint_9/Data/travel_insurance_us.csv', sep =',' )
print(data.info())
print()
print('===='*30)
print(data.dtypes) # to see data types
data.columns = data.columns.str.lower()
print(data.columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50660 entries, 0 to 50659
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Agency                 50660 non-null  object 
 1   Agency Type            50660 non-null  object 
 2   Distribution Channel   50660 non-null  object 
 3   Product Name           50660 non-null  object 
 4   Claim                  50660 non-null  int64  
 5   Duration               50660 non-null  int64  
 6   Destination            50660 non-null  object 
 7   Net Sales              50660 non-null  float64
 8   Commission (in value)  50660 non-null  float64
 9   Gender                 14569 non-null  object 
 10  Age                    50660 non-null  int64  
dtypes: float64(2), int64(3), object(6)
memory usage: 4.3+ MB
None

Agency                    object
Agency Type               object
Distribution Channel      object
Product Name              object
Claim          

*3.Split the data into two sets:*

training set (train)
validation set (valid) — 25% of the source data
Specify random_state=12345. Declare four variables and store the features and target as follows:

features: features_train, features_valid
target: target_train, target_valid
Print the sizes of tables stored in the variables features_train and features_valid.

In [12]:
#Spliting the data into two sets
train , valid = train_test_split(data, test_size =0.25, random_state=12345)
# Create Features and target
# Remove target column
X_train = data.drop('claim', axis= 1)
X_valid = data.drop('claim', axis= 1)
# Create target (the column we  want to predict)
y_train = data['claim']
# Create target (the column we  want to predict)
y_vaild = data['claim']

print(X_train.shape)
print()
print(X_valid.shape)


(50660, 10)

(50660, 10)


**Note**
The usual progression is:
`train`
Used to teach the model.
`valid`
Used during development to evaluate and improve the model.
`test`
Used once at the very end for the final unbiased result.

* If you only split into two sets, people often choose:

train and valid while building the model
train and test when they are doing a simpler exercise or when no final holdout set is required 

In [13]:
# Pilot Train
# Check the unique value of 'agency type' and 'gender'
print(data['agency type'].unique())
print()
print(data['gender'].unique())



['Airlines' 'Travel Agency']

['M' nan 'F']


In [14]:
#  One-Hot Encoding (OHE)
#is a process that allows us to convert a categorical feature
# into numerical form in two steps: 1- Creating a dummy variable for each unique categorical value.
#2- filling in dummy columns with the values from the original catergorical feature
# The dummy variables are Boolean — 1 is assigned if the category matches the observation; otherwise, 0 is assigned.

# OHE in pandas 

dummy_variables = pd.get_dummies(data['gender'], dummy_na=True).astype(int)

gender_dummies = pd.get_dummies(data['gender'].fillna('None'), prefix='gender').astype(int)
print(dummy_variables.head())
print(gender_dummies)


   F  M  NaN
0  0  1    0
1  0  0    1
2  0  0    1
3  0  0    1
4  0  1    0
       gender_F  gender_M  gender_None
0             0         1            0
1             0         0            1
2             0         0            1
3             0         0            1
4             0         1            0
...         ...       ...          ...
50655         0         0            1
50656         0         0            1
50657         0         0            1
50658         0         0            1
50659         0         1            0

[50660 rows x 3 columns]


**`pd.get_dummies() does not create a separate column for missing values unless you ask for it.`**

## What is happening:

* print(data['gender'].unique()) can show None/nan
* but pd.get_dummies(data['gender']) drops missing by default (dummy_na=False)

*If want the missing category to be named "None" instead of nan, do:*
`gender_dummies = pd.get_dummies(data['gender'].fillna('None'), prefix='gender')`

*So the key is: use dummy_na=True (or fillna('None')) before one-hot encoding.*

`Note`

That behavior is normal. In newer pandas versions, pd.get_dummies() often returns bool columns (True/False) instead of integer columns (1/0). They mean the same thing:

`True == 1`

`False == 0`

`To avoide dummy trap :` 

* The pd.get_dummies() function provides us with an easy way to drop a column using the 
drop_first parameter. 
* Passing drop_first=True removes the first column.



====================================================================

*`Task`*

Call pd.get_dummies() to transform the Agency Type column using OHE. Store the transformation result in the agency_ohe variable. Print the first five rows of the transformed table

In [15]:
# Getting dummy for agency Type
agency_ohe_variables = pd.get_dummies(data['agency type'], prefix ='agency').astype(int)
print(agency_ohe_variables.head())


   agency_Airlines  agency_Travel Agency
0                1                     0
1                0                     1
2                0                     1
3                0                     1
4                1                     0


=====================================================================================
`Dummy Trap Task `
1. Encode the Gender column with OHE. Call pd.get_dummies() with the drop_first argument to avoid the dummy trap. Print the first five rows of the modified table. 
 
2. One-hot encode the whole DataFrame. Call pd.get_dummies() with the drop_first argument. Store the encoded DataFrame to the data_ohe variable.

Print the first three rows of the resulting table.

3.Now you've learned what you need to train the logistic regression model we previously failed to fit. Here's what we need to do:

In task 2 you used OHE on the entire DataFrame and stored it to the data_ohe variable. This DataFrame contains the targets to train the model. Your goal is to extract and store them to the targets variable. Remember, the targets are contained in the Claim column.
Initialize the LogisticRegression model with the following parameters random_state=12345 and solver='liblinear'. Store the initialized model to the model variable. Fit the model with the training data (features_train, target_train).

==================================================================================






In [16]:
# 1-dummy for gender: Encode the Gender column with OHE

gender_dummies_drop = pd.get_dummies(data['gender'], drop_first= True, dummy_na=True ).astype(int)

print(gender_dummies_drop.head())

# section 2 : One-hot encode the whole DataFrame
data_dummy = pd.get_dummies(data, drop_first= True, dummy_na=True ).astype(int)

print()
print(data_dummy.head(3))

# section3 
features =data_dummy.drop('claim',axis=1)
target =data_dummy['claim']

X_train,X_valid,y_train ,y_vaild = train_test_split(features, target, random_state= 12345, test_size=0.25 )
model = LogisticRegression(random_state= 12345, solver = 'liblinear')
model.fit(X_train,y_train)
print()
print('Trained')




   M  NaN
0  1    0
1  0    1
2  0    1
3  0    1
4  1    0

   claim  duration  net sales  commission (in value)  age  agency_ART  \
0      0        12         45                     15   39           0   
1      0        50         22                      0   36           0   
2      0       251         80                      0   36           0   

   agency_C2B  agency_CBH  agency_CCR  agency_CSR  ...  destination_URUGUAY  \
0           0           0           0           0  ...                    0   
1           0           0           0           0  ...                    0   
2           0           0           0           0  ...                    0   

   destination_UZBEKISTAN  destination_VANUATU  destination_VENEZUELA  \
0                       0                    0                      0   
1                       0                    0                      0   
2                       0                    0                      0   

   destination_VIET NAM  destination


Trained


=============================================================================

**'Analyis'**

M = 1, NaN = 0 means Gender is M
M = 0, NaN = 1 means Gender is missing
M = 0, NaN = 0 would mean Gender is F

===============================================================================

### **Label Encoding**

In [17]:
# Label Encoding: Assign meaningful numbers that respect the natural ordering
# import OrdinalEncoder from sklearn.preprocessing 
from sklearn.preprocessing import OrdinalEncoder

# we create an instance of the OrdinalEncoder class:
encoder = OrdinalEncoder()
# Before the encoder can be used for transformation, 
# it must be introduced to our data. To do that, 
# we call the fit() method and pass in our categorical data

# two categorical columns from our dataset —
#  Agency and Agency Type — and pass them through the fit() method:
columns_for_encoding = ["agency", "agency type"] # identify columns
two_categorial_features = data.loc[:, columns_for_encoding] # select columns from datasets
encoder.fit(two_categorial_features) # pass them the fit method
# we want to use the transform() **method to transform the categorical data:

encoded_data = encoder.transform(two_categorial_features)
print(encoded_data)
print()
print(type(encoded_data))

# we can call the fit_transform() method instead of calling fit() and transform() separately. 

# We passed in the DataFrame but got back the numpy.ndarray. If we want to get back to the DataFrame, we can initiate a new one like so:

encoded_data_df = pd.DataFrame(data=encoded_data, columns=columns_for_encoding)
print(encoded_data_df.head())






[[9. 0.]
 [7. 1.]
 [7. 1.]
 ...
 [7. 1.]
 [7. 1.]
 [2. 0.]]

<class 'numpy.ndarray'>
   agency  agency type
0     9.0          0.0
1     7.0          1.0
2     7.0          1.0
3     7.0          1.0
4     9.0          0.0


# Tasks for label Encoding 

1.Encode the Distribution Channel column in the dataset. To do that, apply LabelEncoder which is used for a single column transformation.

Store the transformed column in the transformed_dchannel variable. Then, create a DataFrame with the transformed column. Store this DataFrame in the transformed_dchannel_df variable.

Print the first five rows of this DataFrame.


2- We started this lesson by training a decision tree algorithm. The training process failed due to the error associated with the categorical features. 

Now, armed with the new encoding technique that we've just learned, let's transform all the categorical features in our dataset and train the decision tree classifier.

In the precode, you will find the categorical_features variable, which lists the names of all the categorical features in the dataset. All categorical columns are extracted from the original dataset and placed in a separate categorical_data variable.

It's now your time to shine! You know that OrdinalEncoder can transform multiple categorical features at once, so you can apply it to the categorical_data variable. 

When transformed, you create a DataFrame with the encoded data. Store this DataFrame in the transformed_data_df variable.


In [ ]:
#1 idetify the colum for encoding 
col_for_encoding = (['distribution channel'])
encoder_label = LabelEncoder()
#2 get the column from dataset
categorial_features = data.loc[:, col_for_encoding]
#3 Fit_transform
encode_data = encoder_label.fit_transform(categorial_features)
encode_data_df = pd.DataFrame(data=encode_data, columns=col_for_encoding)
print(encode_data_df.head()) 

# task 2 
categorical_features = [
    "agency",
    "agency type",
    "distribution channel",
    "product name",
    "destination",
    "gender",]

encoder_2 = OrdinalEncoder()
data_encoded = data.copy()
data_encoded[categorical_features] = encoder_2.fit_transform(data_encoded[categorical_features])

targets = data_encoded["claim"]
features = data_encoded.drop("claim", axis=1)

X_train, X_valid, y_train, y_valid = train_test_split(
    features, targets, test_size=0.25, random_state=12345
)

tree = DecisionTreeClassifier(random_state=12345)
tree.fit(X_train, y_train)

print("Trained!")

/home/susan/Feature_Engineering_Sprint_9/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


   distribution channel
0                     1
1                     1
2                     1
3                     1
4                     1
Trained!


**Ordinal Encoding**

In [26]:
# Its mapping parameter is better documented and a lot more intuitive to use
#temperature_dict = {'cold': 0, 'warm': 1, 'hot': 2}
#df['temperature'] = df['temperature'].map(temperature_dict)
